In [3]:
import torch
torch.set_printoptions(sci_mode=False)

In [56]:
inputs = torch.randn(6, 3)  # 6 samples, each vector length 3
print(inputs, inputs.shape)

tensor([[ 1.2311, -1.0973, -1.1541],
        [ 0.7763, -0.2582, -2.0407],
        [-0.8016, -0.8183, -1.1820],
        [-0.2877,  1.1031,  1.3334],
        [-1.4053, -0.5922, -0.2548],
        [ 1.1517, -0.0179,  0.4264]]) torch.Size([6, 3])


In [6]:
input_query = inputs[1]
print(input_query)

tensor([-1.1279, -0.2866, -0.4242])


In [7]:
input_1 = inputs[0]

In [8]:
input_query@input_1

tensor(1.0956)

In [9]:
attn_weights = []
for idx, tensor in enumerate(inputs):
    attn_weights.append((inputs[1]@tensor).item())

attn_weights

[1.0956023931503296,
 1.5343255996704102,
 -0.20174464583396912,
 0.3750362694263458,
 0.21933522820472717,
 2.4465622901916504]

In [10]:
# Normalization using softmax
def softmax_naive(x):
    return torch.exp(x)/ (torch.exp(x).sum(dim=0))

attn_weights = softmax_naive(torch.tensor(attn_weights))
print(attn_weights)

tensor([0.1318, 0.2044, 0.0360, 0.0641, 0.0549, 0.5089])


In [11]:
torch.softmax(torch.tensor(attn_weights), dim=0)

C:\Users\Admin\AppData\Local\Temp\ipykernel_12680\175425178.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.softmax(torch.tensor(attn_weights), dim=0)


tensor([0.1587, 0.1706, 0.1442, 0.1483, 0.1469, 0.2313])

In [12]:
context_vector = torch.zeros(inputs[1].shape)
for idx, x_i in enumerate(inputs):
    context_vector = context_vector + attn_weights[idx]*x_i

print(context_vector)


tensor([-1.0100,  0.1406, -1.4918])


 A simple self attention mechanism without trainable weights

In [13]:
attn_scores = torch.empty(6,6)
for i, x_i in enumerate(inputs):
     for j, x_j in enumerate(inputs):
        attn_scores[i, j] = x_i@x_j
print(attn_scores)
# These are attn_scores, not yet normalized

tensor([[ 0.9016,  1.0956,  0.0940, -0.2077, -0.2304,  1.0307],
        [ 1.0956,  1.5343, -0.2017,  0.3750,  0.2193,  2.4466],
        [ 0.0940, -0.2017,  2.2330,  0.3636, -1.1184, -2.4046],
        [-0.2077,  0.3750,  0.3636,  3.0580,  1.3469,  2.9391],
        [-0.2304,  0.2193, -1.1184,  1.3469,  1.3452,  2.7935],
        [ 1.0307,  2.4466, -2.4046,  2.9391,  2.7935,  8.4490]])


In [14]:
attn_scores_ = inputs@inputs.T

In [15]:
# Normalization
attn_weights = torch.softmax(attn_scores, dim=1)
attn_weights_ = torch.softmax(attn_scores_, dim=1)
print(attn_weights)
print(attn_weights_)

tensor([[0.2247, 0.2728, 0.1002, 0.0741, 0.0724, 0.2557],
        [0.1318, 0.2044, 0.0360, 0.0641, 0.0549, 0.5089],
        [0.0839, 0.0624, 0.7121, 0.1098, 0.0249, 0.0069],
        [0.0170, 0.0305, 0.0301, 0.4459, 0.0806, 0.3959],
        [0.0301, 0.0472, 0.0124, 0.1457, 0.1455, 0.6191],
        [0.0006, 0.0024, 0.0000, 0.0040, 0.0035, 0.9895]])
tensor([[0.2247, 0.2728, 0.1002, 0.0741, 0.0724, 0.2557],
        [0.1318, 0.2044, 0.0360, 0.0641, 0.0549, 0.5089],
        [0.0839, 0.0624, 0.7121, 0.1098, 0.0249, 0.0069],
        [0.0170, 0.0305, 0.0301, 0.4459, 0.0806, 0.3959],
        [0.0301, 0.0472, 0.0124, 0.1457, 0.1455, 0.6191],
        [0.0006, 0.0024, 0.0000, 0.0040, 0.0035, 0.9895]])


In [16]:
attn_weights.shape

torch.Size([6, 6])

In [17]:
context_vec = attn_weights@inputs
print(attn_weights.shape)
print(inputs.shape)
print(context_vec)

print(attn_weights[0]@inputs)

torch.Size([6, 6])
torch.Size([6, 3])
tensor([[-0.7910, -0.1440, -0.8966],
        [-1.0100,  0.1406, -1.4918],
        [ 0.1544, -1.0816,  0.0667],
        [-0.3936, -0.0660, -1.7682],
        [-0.8619,  0.3077, -1.9336],
        [-1.3722,  0.5864, -2.4700]])
tensor([-0.7910, -0.1440, -0.8966])


In [18]:
# Implementing the attention weights with trainable weights
inputs
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [19]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out))
W_key = torch.nn.Parameter(torch.rand(d_in, d_out))
W_value = torch.nn.Parameter(torch.rand(d_in, d_out))

In [20]:
W_query, W_key, W_value

(Parameter containing:
 tensor([[0.2961, 0.5166],
         [0.2517, 0.6886],
         [0.0740, 0.8665]], requires_grad=True),
 Parameter containing:
 tensor([[0.1366, 0.1025],
         [0.1841, 0.7264],
         [0.3153, 0.6871]], requires_grad=True),
 Parameter containing:
 tensor([[0.0756, 0.1966],
         [0.3164, 0.4017],
         [0.1186, 0.8274]], requires_grad=True))

In [21]:
query_2 = x_2 @ W_query
x_2, query_2

(tensor([-1.1279, -0.2866, -0.4242]),
 tensor([-0.4375, -1.1476], grad_fn=<SqueezeBackward4>))

In [22]:
keys = inputs@W_key
values = inputs@W_value

print(keys.shape, values.shape)

torch.Size([6, 2]) torch.Size([6, 2])


In [23]:
keys, values

(tensor([[-0.1741, -0.3022],
         [-0.3405, -0.6152],
         [-0.0681, -0.6698],
         [-0.5559, -1.5026],
         [-0.2270, -0.3452],
         [-0.8633, -1.4178]], grad_fn=<MmBackward0>),
 tensor([[-0.1621, -0.2866],
         [-0.2263, -0.6879],
         [-0.3584, -0.1257],
         [-0.3575, -1.4749],
         [ 0.0404, -0.6478],
         [-0.2114, -2.0899]], grad_fn=<MmBackward0>))

In [24]:
attn_scores_2 = query_2 @ keys.T   # query_2(1x3) @ keys.T(3x6)
attn_scores_2

tensor([0.4229, 0.8550, 0.7984, 1.9675, 0.4954, 2.0047],
       grad_fn=<SqueezeBackward4>)

In [25]:
d_k = keys.shape[1]  
attn_weights_2 = torch.softmax(attn_scores_2/d_k**0.5, dim=0)
print(attn_weights_2, attn_weights_2.shape)


tensor([0.0930, 0.1262, 0.1213, 0.2772, 0.0979, 0.2845],
       grad_fn=<SoftmaxBackward0>) torch.Size([6])


In [26]:
context_vec_2 = attn_weights_2 @ values # attn_weights_2(6) @ values(6,2)
context_vec_2

tensor([-0.2424, -1.1955], grad_fn=<SqueezeBackward4>)

In [27]:
# Implementing a genrelized class
import torch.nn as nn
class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_in = d_in
        self.d_out = d_out
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))
        self.div = self.d_out**0.5

    def forward(self, x):   # x(6,3) all for example only
        query = x @ self.W_query  #query(6,2)  
        key = x @ self.W_key      #key(6,2)
        value = x @ self.W_value  #value(6,2)

        # Need to calculate the attention scores
        attn_scores = query @ key.T   # (6,2) x (2,6) = (6,6)
        attn_weigths = torch.softmax(attn_scores/self.div, dim = 1) # (6,6)
        context_vec = attn_weigths @ value #(6,2)
        return context_vec, attn_weigths

In [28]:
sa_ver1 = SelfAttention(3,2)

In [29]:
inputs

tensor([[-0.8999, -0.3028,  0.0145],
        [-1.1279, -0.2866, -0.4242],
        [ 0.3676, -1.3825,  0.4320],
        [ 0.4214, -0.6419, -1.5711],
        [ 0.0705,  0.5017, -1.0433],
        [-1.3854,  0.5944, -2.4852]])

In [30]:
CV, AW= sa_ver1(inputs)
print(CV)
print(AW)
print(AW.sum())

tensor([[-1.6935, -1.0460],
        [-1.9011, -1.1535],
        [-1.7914, -1.0525],
        [-2.1677, -1.2693],
        [-1.4700, -0.9536],
        [-2.2051, -1.3117]], grad_fn=<MmBackward0>)
tensor([[0.1419, 0.1863, 0.1237, 0.1746, 0.0863, 0.2872],
        [0.1235, 0.1931, 0.0998, 0.1645, 0.0500, 0.3690],
        [0.1107, 0.1437, 0.0887, 0.1957, 0.1236, 0.3375],
        [0.0908, 0.1692, 0.0654, 0.1547, 0.0319, 0.4880],
        [0.1729, 0.1991, 0.1684, 0.1600, 0.0996, 0.2001],
        [0.0934, 0.2162, 0.0677, 0.1157, 0.0102, 0.4968]],
       grad_fn=<SoftmaxBackward0>)
tensor(6., grad_fn=<SumBackward0>)


In [31]:
import torch
import torch.nn as nn

class SelfAttentionVer2(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()

        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key   = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)

        self.scale = d_out ** 0.5

    def forward(self, x):

        query = self.W_query(x)
        key   = self.W_key(x)
        value = self.W_value(x)

        attn_scores = query @ key.T
        attn_weights = torch.softmax(attn_scores / self.scale, dim=1)

        context_vec = attn_weights @ value

        return context_vec, attn_weights

In [32]:
sa_ver2 = SelfAttentionVer2(3,2)
CV, AW= sa_ver2(inputs)
print(CV)
print(AW)
print(AW.sum())

tensor([[-0.5011, -0.2947],
        [-0.5101, -0.3031],
        [-0.4760, -0.2714],
        [-0.4937, -0.2867],
        [-0.4895, -0.2833],
        [-0.5305, -0.3231]], grad_fn=<MmBackward0>)
tensor([[0.1626, 0.1649, 0.1289, 0.1528, 0.1889, 0.2019],
        [0.1602, 0.1631, 0.1123, 0.1439, 0.2007, 0.2198],
        [0.1674, 0.1666, 0.1784, 0.1698, 0.1605, 0.1572],
        [0.1680, 0.1676, 0.1384, 0.1512, 0.1868, 0.1880],
        [0.1672, 0.1672, 0.1488, 0.1579, 0.1785, 0.1803],
        [0.1518, 0.1556, 0.0764, 0.1187, 0.2314, 0.2662]],
       grad_fn=<SoftmaxBackward0>)
tensor(6., grad_fn=<SumBackward0>)


In [33]:
# Causal Attention Masks
import torch
import torch.nn as nn

class SelfAttentionVer3(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()

        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key   = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)

        self.scale = d_out ** 0.5

    def forward(self, x):

        query = self.W_query(x)
        key   = self.W_key(x)
        value = self.W_value(x)

        attn_scores = query @ key.T
        seq_len = x.shape[0]
        causal_mask = torch.tril(torch.ones(seq_len, seq_len, device=attn_scores.device))
        attn_scores = attn_scores.masked_fill(causal_mask == 0, float('-inf'))
        attn_weights = torch.softmax(attn_scores / self.scale, dim=1)
        

        context_vec = attn_weights @ value

        return context_vec, attn_weights

In [34]:
sa_ver2 = SelfAttentionVer3(3,2)
CV, AW= sa_ver2(inputs)
print(CV)
print(AW)
print(AW.sum())

tensor([[ 0.1322, -0.3916],
        [ 0.2735, -0.4980],
        [ 0.1445, -0.4861],
        [ 0.3909, -0.5766],
        [ 0.3467, -0.4652],
        [ 0.7495, -0.6003]], grad_fn=<MmBackward0>)
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4893, 0.5107, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3474, 0.3786, 0.2740, 0.0000, 0.0000, 0.0000],
        [0.2492, 0.3009, 0.1451, 0.3048, 0.0000, 0.0000],
        [0.2144, 0.2241, 0.1877, 0.1895, 0.1843, 0.0000],
        [0.1307, 0.1495, 0.0893, 0.1636, 0.1749, 0.2920]],
       grad_fn=<SoftmaxBackward0>)
tensor(6., grad_fn=<SumBackward0>)


In [57]:
# Creating a batch of inputs
inputs_batch = torch.stack((inputs, inputs), dim=0)

In [76]:
# Masking Additional Attentionn Weights With Dropout
# Causal Attention Masks
import torch
import torch.nn as nn

class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length ,dropout_factor=0):
        super().__init__()

        self.W_query = nn.Linear(d_in, d_out, bias=False) # (3,2), (d_in, d_out)
        self.W_key   = nn.Linear(d_in, d_out, bias=False) # (3,2), (d_in, d_out)
        self.W_value = nn.Linear(d_in, d_out, bias=False) # (3,2), (d_in, d_out)
        self.dropout = nn.Dropout(dropout_factor)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))
        self.scale = d_out ** 0.5

    def forward(self, x):

        print("Inside Causal Attention")
        query = self.W_query(x) # (Batch, 6,2) , (batch, context_length, d_out)
        key   = self.W_key(x) # (Batch, 6,2) , (batch, context_length, d_out)
        value = self.W_value(x) # (Batch, 6,2) , (batch, context_length, d_out)

        # Q.KT
        attn_scores = query @ key.transpose(1, 2)   # (Batch, 6,6) = (Batch, 6,2) x (Batch, 6,2), but if we have a batch, we need to make the suitable changes in the multiplication
        # Masking
        attn_scores.masked_fill_(self.mask == 1, float('-inf'))  # (Batch, 6,6) 
        # Softmax((Q.KT)/root(d_out))
        attn_weights = torch.softmax(attn_scores / self.scale, dim=2)  # (Batch, 6,6) 
        # Softmax((Q.KT)/root(d_out)).V
        context_vec = attn_weights @ value  # (Batch, 6,2) = (Batch, 6,6) x (Batch, 6,2)

        return context_vec, attn_weights

In [49]:
batch, context_length, embed_dim = inputs_batch.shape
d_in = embed_dim
d_out = 2
CausalAttention = CausalAttention(d_in, d_out, context_length)

In [50]:
context_vec, attn_weights = CausalAttention(inputs_batch)
print(context_vec)
print(attn_weights)

tensor([[[ 0.0515,  0.1276],
         [ 0.0713,  0.0364],
         [ 0.1435,  0.0370],
         [ 0.2346, -0.2196],
         [ 0.2034, -0.2338],
         [ 0.2361, -0.1783]],

        [[ 0.0515,  0.1276],
         [ 0.0713,  0.0364],
         [ 0.1435,  0.0370],
         [ 0.2346, -0.2196],
         [ 0.2034, -0.2338],
         [ 0.2361, -0.1783]]], grad_fn=<UnsafeViewBackward0>)
tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5349, 0.4651, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3494, 0.4468, 0.2038, 0.0000, 0.0000, 0.0000],
         [0.2477, 0.3008, 0.1777, 0.2737, 0.0000, 0.0000],
         [0.2026, 0.1954, 0.2295, 0.1904, 0.1820, 0.0000],
         [0.1830, 0.1408, 0.3765, 0.1325, 0.1239, 0.0434]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5349, 0.4651, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3494, 0.4468, 0.2038, 0.0000, 0.0000, 0.0000],
         [0.2477, 0.3008, 0.1777, 0.2737, 0.0000, 0.0000],
         [0.2026, 0.1954,

In [77]:
# MultiHead Causal Attention
# Stacking Multiple Single Head Attention Blocks
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length ,dropout_factor=0, num_heads=2):
        super().__init__()

        self.heads = nn.ModuleList([
            CausalAttention(d_in, d_out, context_length ,dropout_factor=0) for _ in range(num_heads)])

    def forward(self, x):
        print("Inside Multi Head Attention")

        contexts = []
        weights = []

        for head in self.heads:
            context, attn = head(x)
            contexts.append(context)
            weights.append(attn)

        context = torch.cat(contexts, dim=-1)

        return context, weights
        
    


In [78]:
batch, context_length, embed_dim = inputs_batch.shape
d_in = embed_dim
d_out = 2
print(batch, context_length, embed_dim, d_in, d_out)

2 6 3 3 2


In [79]:
SimpleMHA = MultiHeadAttention(d_in, d_out, context_length ,dropout_factor=0, num_heads=2)

In [80]:
SimpleMHA(inputs_batch)

Inside Multi Head Attention
Inside Causal Attention
Inside Causal Attention


(tensor([[[-0.0957,  0.2084,  0.4594, -1.1400],
          [-0.3177,  0.2095,  0.7755, -1.0171],
          [-0.1576,  0.2112,  0.3825, -0.8192],
          [-0.1020,  0.1060,  0.2057, -0.2717],
          [-0.0192,  0.1190, -0.1111, -0.1209],
          [ 0.0346,  0.0607,  0.2670, -0.4435]],
 
         [[-0.0957,  0.2084,  0.4594, -1.1400],
          [-0.3177,  0.2095,  0.7755, -1.0171],
          [-0.1576,  0.2112,  0.3825, -0.8192],
          [-0.1020,  0.1060,  0.2057, -0.2717],
          [-0.0192,  0.1190, -0.1111, -0.1209],
          [ 0.0346,  0.0607,  0.2670, -0.4435]]], grad_fn=<CatBackward0>),
 [tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
           [0.5313, 0.4687, 0.0000, 0.0000, 0.0000, 0.0000],
           [0.3261, 0.3925, 0.2813, 0.0000, 0.0000, 0.0000],
           [0.2355, 0.2532, 0.2784, 0.2329, 0.0000, 0.0000],
           [0.1973, 0.2676, 0.1991, 0.1665, 0.1695, 0.0000],
           [0.1567, 0.1186, 0.1672, 0.1853, 0.1974, 0.1748]],
  
          [[1.0000, 0.00

In [111]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.d_in = d_in
        self.d_out = d_out
        self.context_length = context_length 
        self.dropout = dropout
        self.num_heads = num_heads
        self.head_dim = self.d_out // self.num_heads
        
        self.W_Query = nn.Linear(self.d_in, self.d_out, bias = qkv_bias)
        self.W_Key = nn.Linear(self.d_in, self.d_out, bias = qkv_bias)
        self.W_Value = nn.Linear(self.d_in, self.d_out, bias = qkv_bias)
        self.out_proj = nn.Linear(self.d_out, self.d_out)
        self.dropout = nn.Dropout(self.dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        batch, num_tokens, d_in = x.shape # (2,6,3)

        # This is to convert the dim_in = 3 to dim_out = 2
        Query = self.W_Query(x)  # Query (2,6,2)
        Key = self.W_Key(x) # Key (2,6,2)
        Value = self.W_Value(x) # Value (2,6,2)

        # Single To Multi Head Transformation
        Query = Query.view(batch, num_tokens, self.num_heads, self.head_dim) # Query (2,6,2, 1)
        Key = Key.view(batch, num_tokens, self.num_heads, self.head_dim) # Key (2,6,2, 1)
        Value = Value.view(batch, num_tokens, self.num_heads, self.head_dim) # Value (2,6,2, 1)

        # Rearrange tensor so each head processes the full token sequence independently.
        # Shape: (batch, tokens, num_heads, head_dim) → (batch, num_heads, tokens, head_dim) 
        Query = Query.transpose(1,2)# Query (2,2,6, 1)
        Key = Key.transpose(1,2)# Key (2,2,6, 1)
        Value = Value.transpose(1,2)# Value (2,2,6, 1)


        # Self Attention 
        attn_scores = Query @ Key.transpose(2,3)  # attn_scores (2,2,6,6) = (2,2,6,1) @ (2,2,1,6)

        # Createing a causal mask to defer lookinh into the future token
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens] # need to fill

        attn_scores.masked_fill_(mask_bool, -torch.inf)# need to fill

        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5, dim=-1)# need to fill
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ Value) # (2,2,6,6) @ (2,2,6,1) >>  (2,2,6,1)

        context_vec = context_vec.transpose(2,1) # (2,2,6,1) >> (2,6,2,1)

        # Need to combine all the information from the heads
        context_vec = context_vec.contiguous().view(batch, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)

        return context_vec

In [112]:
torch.manual_seed(123)
batch, context_length, embed_dim = inputs_batch.shape
print("Batch : ", batch, "\nContext_length : ", context_length, "\nEmbed_Dim : ", embed_dim)


Batch :  2 
Context_length :  6 
Embed_Dim :  3


In [113]:
d_in = embed_dim
d_out = 4
print("D_in : ", d_in, "\nD_out : ", d_out)

D_in :  3 
D_out :  4


In [114]:
MHA = MultiHeadAttention(d_in, d_out, context_length, dropout=0.0, num_heads=2)

In [115]:
MHA(inputs_batch)

tensor([[[ 0.1148, -0.0210, -0.0585, -0.3898],
         [-0.0175, -0.0603, -0.0411, -0.1441],
         [ 0.1093, -0.0426, -0.0430, -0.3103],
         [ 0.1173,  0.0702, -0.0472, -0.3679],
         [ 0.1153,  0.1008, -0.0576, -0.4069],
         [ 0.1411,  0.1322, -0.0573, -0.4519]],

        [[ 0.1148, -0.0210, -0.0585, -0.3898],
         [-0.0175, -0.0603, -0.0411, -0.1441],
         [ 0.1093, -0.0426, -0.0430, -0.3103],
         [ 0.1173,  0.0702, -0.0472, -0.3679],
         [ 0.1153,  0.1008, -0.0576, -0.4069],
         [ 0.1411,  0.1322, -0.0573, -0.4519]]], grad_fn=<ViewBackward0>)